# Embeddings reduceren naar 2D met UMAP

Draait in Snowflake. Leest de embeddings uit `FRUTA.PUBLIC.RECORD_EMBEDDINGS`, reduceert ze met UMAP naar twee dimensies en schrijft de coördinaten naar `FRUTA.PUBLIC.RECORD_UMAP_2D` (`NOTE_ID`, `X`, `Y`).
De app leest alleen die tabel; `umap-learn` en het model zijn in de app niet nodig.

Vereist het pakket `umap-learn`: voeg het toe via *Packages* (warehouse-runtime) of installeer het met de cel hieronder (container-runtime).

In [ ]:
# Alleen nodig op container-runtime
!pip install umap-learn

In [ ]:
import json

import numpy as np
from snowflake.snowpark.context import get_active_session
from umap import UMAP

session = get_active_session()

EMBEDDINGS_TABLE = "FRUTA.PUBLIC.RECORD_EMBEDDINGS"
UMAP_TABLE = "FRUTA.PUBLIC.RECORD_UMAP_2D"

UMAP_PARAMS = {
    "n_components": 2,  # Reduceert naar 2D voor visualisatie. 3D is ook mogelijk, maar moeilijker te interpreteren
    "n_neighbors": 12,  # Default: 15. Lagere waarden leiden tot meer lokale structuren, hogere waarden tot meer globale
    "min_dist": 0.05,  # Default: 0.1. Lagere waarde = dichtere clusters en meer gescheiden groepen
    "random_state": 6,  # Zorgt voor reproduceerbaarheid
}

In [ ]:
# Haal de embeddings op; VECTOR als ARRAY ophalen, dat komt in pandas binnen als JSON-tekst
df = session.sql(
    f"SELECT NOTE_ID, EMBEDDING::ARRAY AS EMBEDDING FROM {EMBEDDINGS_TABLE}"
).to_pandas()
embs = np.array(
    [json.loads(e) if isinstance(e, str) else e for e in df["EMBEDDING"]],
    dtype=np.float32,
)
print(f"{len(df)} embeddings opgehaald, vorm {embs.shape}.")

In [ ]:
# Train UMAP en reduceer naar 2D
reducer = UMAP(**UMAP_PARAMS)
reduced_embs = reducer.fit_transform(embs)

In [ ]:
# Schrijf de coördinaten naar Snowflake
df_umap = df[["NOTE_ID"]].copy()
df_umap["X"] = reduced_embs[:, 0]
df_umap["Y"] = reduced_embs[:, 1]

session.write_pandas(
    df_umap,
    "RECORD_UMAP_2D",
    database="FRUTA",
    schema="PUBLIC",
    auto_create_table=True,
    overwrite=True,
    quote_identifiers=False,
)
print(f"Tabel '{UMAP_TABLE}' geschreven: {len(df_umap)} rijen.")

In [ ]:
# Controle: aantal embeddings en aantal coördinaten
session.sql(
    f"""
    SELECT
        (SELECT COUNT(*) FROM {EMBEDDINGS_TABLE}) AS EMBEDDINGS,
        (SELECT COUNT(*) FROM {UMAP_TABLE}) AS COORDINATEN
    """
).show()